In [1]:
import os
import cv2
import random
import shutil
from pathlib import Path
from tqdm import tqdm

random.seed(42)

DATASET_ROOT = "datasets"
PREPARED_ROOT = "prepared_data"

os.makedirs(PREPARED_ROOT, exist_ok=True)

In [2]:
FF_VIDEO_DIR = f"{DATASET_ROOT}/FF"
FF_FRAME_DIR = f"{DATASET_ROOT}/FF_frames"

FRAME_GAP = 10   # take 1 frame every 10 frames

def extract_frames(video_path, save_dir):
    cap = cv2.VideoCapture(video_path)
    frame_id, saved = 0, 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_id % FRAME_GAP == 0:
            filename = f"{Path(video_path).stem}_{saved}.jpg"
            cv2.imwrite(os.path.join(save_dir, filename), frame)
            saved += 1

        frame_id += 1
    cap.release()

for label in ["real", "fake"]:
    src = os.path.join(FF_VIDEO_DIR, label)
    dst = os.path.join(FF_FRAME_DIR, label)
    os.makedirs(dst, exist_ok=True)

    videos = [v for v in os.listdir(src) if v.endswith((".mp4",".avi",".mov"))]

    for vid in tqdm(videos, desc=f"Extracting {label}"):
        extract_frames(os.path.join(src, vid), dst)

print("FF frames extracted")

Extracting fake: 100%|██████████| 200/200 [11:59<00:00,  3.60s/it]

FF frames extracted


In [3]:
GLOBAL_DIR = f"{PREPARED_ROOT}/global_test"
os.makedirs(f"{GLOBAL_DIR}/real", exist_ok=True)
os.makedirs(f"{GLOBAL_DIR}/fake", exist_ok=True)

SAMPLE_RATIO = 0.10

def sample_images(folder, label):
    files = os.listdir(folder)
    k = max(1, int(len(files)*SAMPLE_RATIO))
    sampled = random.sample(files, k)

    for f in sampled:
        shutil.copy(
            os.path.join(folder,f),
            os.path.join(GLOBAL_DIR,label,f"{Path(folder).stem}_{f}")
        )

# CASIA (color only)
sample_images(f"{DATASET_ROOT}/CASIA-FASD/train_img/train_img/color","fake")
sample_images(f"{DATASET_ROOT}/CASIA-FASD/test_img/test_img/color","fake")

# SiW
sample_images(f"{DATASET_ROOT}/SiW/train/real","real")
sample_images(f"{DATASET_ROOT}/SiW/train/spoof","fake")

# FF frames
sample_images(f"{DATASET_ROOT}/FF_frames/real","real")
sample_images(f"{DATASET_ROOT}/FF_frames/fake","fake")
print("Global dataset created")

Global dataset created


In [4]:
CASIA_DST = f"{PREPARED_ROOT}/client1_casia"

def prepare_casia_split(src, split):
    for label in ["real","fake"]:
        os.makedirs(f"{CASIA_DST}/{split}/{label}", exist_ok=True)

    for img in tqdm(os.listdir(src), desc=f"CASIA {split}"):
        label = "real" if "real" in img.lower() else "fake"
        shutil.copy(
            os.path.join(src,img),
            os.path.join(CASIA_DST,split,label,img)
        )

prepare_casia_split(f"{DATASET_ROOT}/CASIA-FASD/train_img/train_img/color","train")
prepare_casia_split(f"{DATASET_ROOT}/CASIA-FASD/test_img/test_img/color","test")

print("CASIA client ready")

CASIA test: 100%|██████████| 2408/2408 [00:27<00:00, 87.54it/s]

CASIA client ready


In [5]:
SIW_DST = f"{PREPARED_ROOT}/client2_siw"

for split in ["train","test"]:
    for label in ["real","fake"]:
        os.makedirs(f"{SIW_DST}/{split}/{label}", exist_ok=True)

def copy_folder(src,label,split):
    for img in os.listdir(src):
        shutil.copy(src+"/"+img, f"{SIW_DST}/{split}/{label}/{img}")

# train = train + val
copy_folder(f"{DATASET_ROOT}/SiW/train/real","real","train")
copy_folder(f"{DATASET_ROOT}/SiW/train/spoof","fake","train")
copy_folder(f"{DATASET_ROOT}/SiW/val/real","real","train")
copy_folder(f"{DATASET_ROOT}/SiW/val/spoof","fake","train")

# test
copy_folder(f"{DATASET_ROOT}/SiW/test/real","real","test")
copy_folder(f"{DATASET_ROOT}/SiW/test/spoof","fake","test")

print("SiW client ready")

SiW client ready


In [7]:
FF_DST = f"{PREPARED_ROOT}/client3_ff"
SRC = f"{DATASET_ROOT}/FF_frames"

for split in ["train","test"]:
    for label in ["real","fake"]:
        os.makedirs(f"{FF_DST}/{split}/{label}", exist_ok=True)

def split_copy(label):
    files = os.listdir(f"{SRC}/{label}")
    random.shuffle(files)

    cut = int(len(files)*0.8)
    train,test = files[:cut],files[cut:]

    for f in train:
        shutil.copy(f"{SRC}/{label}/{f}",f"{FF_DST}/train/{label}/{f}")

    for f in test:
        shutil.copy(f"{SRC}/{label}/{f}",f"{FF_DST}/test/{label}/{f}")

split_copy("real")
split_copy("fake")

print("FF client ready")

FF client ready


In [8]:
def count_images(path):
    total=0
    for root,_,files in os.walk(path):
        total+=len(files)
    return total

for folder in os.listdir(PREPARED_ROOT):
    print(folder, "->", count_images(f"{PREPARED_ROOT}/{folder}"), "images")

client1_casia -> 4063 images
client2_siw -> 7586 images
client3_ff -> 31310 images
global_test -> 4129 images
